In [1]:
#Necessary packages to load data
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from minisom import MiniSom
import sklearn_som
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from sammon import sammon

In [2]:
base_dir =r"D:\ERA 5 Data\ERA5_Camden"

In [ ]:
var_config = {
    "gph": {
        "folder": "Geopotential_500hPa",
        "pattern": "era5_geopotential_500hPa__*.nc"
    },
}

In [4]:
#Combine each year
def load_variable(var_name):
    config = var_config[var_name]

    path = os.path.join(base_dir, config["folder"], config["pattern"])

    ds = xr.open_mfdataset(
        path,
        combine="by_coords",
        parallel=False   # IMPORTANT (Windows stability)
    )

    return ds

In [5]:
#To look at other variables raw data change what is in "" to any of the variables in block 3
ds = load_variable("gph")

print(ds)               # full structure
print(ds.variables)     # variables
print(ds.dims)          # dimensions
print(ds.attrs)         # metadata

<xarray.Dataset> Size: 12GB
Dimensions:         (valid_time: 192864, pressure_level: 1, latitude: 97,
                     longitude: 161)
Coordinates:
    number          int64 8B 0
  * valid_time      (valid_time) datetime64[ns] 2MB 2004-01-01 ... 2025-12-31...
  * pressure_level  (pressure_level) float64 8B 500.0
  * latitude        (latitude) float64 776B 54.0 53.75 53.5 ... 30.5 30.25 30.0
  * longitude       (longitude) float64 1kB -100.0 -99.75 -99.5 ... -60.25 -60.0
    expver          (valid_time) <U4 3MB dask.array<chunksize=(8784,), meta=np.ndarray>
Data variables:
    z               (valid_time, pressure_level, latitude, longitude) float32 12GB dask.array<chunksize=(2928, 1, 33, 54), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    

In [6]:
#Enter in the actual data variable title from the ERA5 data
ds.z

<xarray.DataArray 'z' (valid_time: 192864, pressure_level: 1, latitude: 97,
                       longitude: 161)> Size: 12GB
dask.array<concatenate, shape=(192864, 1, 97, 161), dtype=float32, chunksize=(2928, 1, 33, 54), chunktype=numpy.ndarray>
Coordinates:
    number          int64 8B 0
  * valid_time      (valid_time) datetime64[ns] 2MB 2004-01-01 ... 2025-12-31...
  * pressure_level  (pressure_level) float64 8B 500.0
  * latitude        (latitude) float64 776B 54.0 53.75 53.5 ... 30.5 30.25 30.0
  * longitude       (longitude) float64 1kB -100.0 -99.75 -99.5 ... -60.25 -60.0
    expver          (valid_time) <U4 3MB dask.array<chunksize=(8784,), meta=np.ndarray>
Attributes: (12/31)
    GRIB_paramId:                             129
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      15617
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           z
    GRIB_totalNumber:                         0
    GRIB_units:                               m**2 s**-2
    long_name:                                Geopotential
    units:                                    m**2 s**-2
    standard_name:                            geopotential

In [7]:
# use variable titles from block 3: "u10", "v10" etc **ORDER MATTERS**
vars_to_use = [
    "gph"
]

#Load datasets
datasets = {var: load_variable(var) for var in vars_to_use}

# Create an empty 2D (Rows & Columns) array to store the subset (May–Sept) + daily max
processed_datasets = {}

# Create a loop to iterate through the datasets and process them
for name, ds in datasets.items():

    var = next(iter(ds.data_vars))

    # May-Sept only with Boolean Mask
    ds_warm = ds.where(
        (ds.valid_time.dt.month >= 5) &
        (ds.valid_time.dt.month <= 9),
        drop=True
    )

    print(name, "after subset:", len(ds_warm.valid_time))

    # Daily aggregation
    if name in ["temp"]:
        ds_daily = (
            ds_warm
            .resample(valid_time="24h")
            .max()
        )

    elif name in ["gph"]:
        ds_daily = (
            ds_warm
            .resample(valid_time="24h")
            .mean()
        )

    ds_daily = ds_daily.dropna(dim="valid_time", how="all")

    processed_datasets[name] = ds_daily

    print(name, "after daily:", len(ds_daily.valid_time))

gph after subset: 80784
gph after daily: 3366


In [8]:
for name, ds in processed_datasets.items():
    print(name, len(ds.valid_time))

print(datasets.keys())
for name, ds in datasets.items():
    print(name, list(ds.data_vars))

gph 3366
dict_keys(['gph'])
gph ['z']


In [16]:
from pathlib import Path

WEATHER_FILE = Path(
    r"D:\ERA 5 Data\ERA5_Camden\daily_weather.nc"
)

# Get the processed geopotential dataset
ds_weather = processed_datasets["gph"]

# Get its actual variable name (it might be "z" rather than "gph")
var_name = next(iter(ds_weather.data_vars))

# Select 500 hPa and preserve the spatial grid
gph = (
    ds_weather[var_name]
    .sel(pressure_level=500, drop=True)
    .transpose("valid_time", "latitude", "longitude")
    .astype("float32")
)

# Save with a consistent variable name
ds_weather = gph.to_dataset(name="gph").sortby("valid_time")

WEATHER_FILE.parent.mkdir(parents=True, exist_ok=True)
ds_weather.to_netcdf(WEATHER_FILE)

print(ds_weather)
print("Saved to:", WEATHER_FILE)

<xarray.Dataset> Size: 210MB
Dimensions:     (latitude: 97, longitude: 161, valid_time: 3366)
Coordinates:
  * latitude    (latitude) float64 776B 54.0 53.75 53.5 ... 30.5 30.25 30.0
  * longitude   (longitude) float64 1kB -100.0 -99.75 -99.5 ... -60.25 -60.0
  * valid_time  (valid_time) datetime64[ns] 27kB 2004-05-01 ... 2025-09-30
    number      int64 8B 0
Data variables:
    gph         (valid_time, latitude, longitude) float32 210MB dask.array<chunksize=(1, 33, 54), meta=np.ndarray>
Saved to: D:\ERA 5 Data\ERA5_Camden\daily_weather.nc
